# Sakila DVD Rental: Combining Tables with pandas

**Author:** Maria Pupysheva · **Date:** October 2026

**Data:** MySQL sample database *sakila*, a fictional DVD rental store (May 2005 – Feb 2006).

**Goal:** practise merging related tables in pandas and answer business questions that need
data from several tables. Results are cross-checked against earlier SQL queries and against
total revenue after every merge.

**Tools:** Python 3.12 · pandas · SQLAlchemy · MySQL 8

**Contents**
1. Setup · 2. Table relationships (Q1) · 3. Revenue by film attributes (Q2–Q3) · 4. Stores (Q4)
· 5. Customers (Q5) · 6. Catalogue coverage (Q6) · 7. Summary

## 1. Setup
Connect to the local MySQL server and load seven tables into pandas DataFrames.
The password is requested at runtime, so it is never stored in the notebook.

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from getpass import getpass

pd.set_option("display.max_columns", 20)   # show all columns in wide tables

In [12]:
engine = create_engine(
    f"mysql+pymysql://root:{getpass('MySQL password: ')}@localhost:3306/sakila"
)

payments      = pd.read_sql("SELECT * FROM payment",       engine)
customers     = pd.read_sql("SELECT * FROM customer",      engine)
films         = pd.read_sql("SELECT * FROM film",          engine)
rentals       = pd.read_sql("SELECT * FROM rental",        engine)
inventory     = pd.read_sql("SELECT * FROM inventory",     engine)
film_category = pd.read_sql("SELECT * FROM film_category", engine)
category      = pd.read_sql("SELECT * FROM category",      engine)

total_revenue = payments["amount"].sum()   # reference value for every sanity check

# Size of each table
tables = {
    "payments": payments, "customers": customers, "films": films, "rentals": rentals,
    "inventory": inventory, "film_category": film_category, "category": category,
}
pd.DataFrame(
    {"rows": [len(df) for df in tables.values()], "columns": [df.shape[1] for df in tables.values()]},
    index=tables.keys(),
)

,rows,columns
payments,16044,7
customers,599,9
films,1000,13
rentals,16044,7
inventory,4581,4
film_category,1000,3
category,16,3


## 2. Table relationships

### Q1. How the tables connect
Before merging, I identified the key column that links each pair of tables.
Merging on the wrong column silently creates wrong or duplicated rows, so this check
comes first. For each pair, I listed the column names the two tables share
(set intersection), then judged which shared column is a real key.

In [13]:
# Columns shared by each pair of tables
pairs = {
    "payments ↔ rentals":       (payments, rentals),
    "rentals ↔ inventory":      (rentals, inventory),
    "inventory ↔ film_category": (inventory, film_category),
    "film_category ↔ category": (film_category, category),
}

for name, (left, right) in pairs.items():
    print(f"{name:28} {sorted(set(left.columns) & set(right.columns))}")

payments ↔ rentals           ['customer_id', 'last_update', 'rental_id', 'staff_id']
rentals ↔ inventory          ['inventory_id', 'last_update']
inventory ↔ film_category    ['film_id', 'last_update']
film_category ↔ category     ['category_id', 'last_update']


**Conclusion:** the keys for the merge chain are

| Tables | Key | Relationship |
|---|---|---|
| payments ↔ rentals | `rental_id` | each payment belongs to one rental |
| rentals ↔ inventory | `inventory_id` | each rental is of one physical DVD copy |
| inventory ↔ film_category | `film_id` | each copy is of one film, and each film has one category |
| film_category ↔ category | `category_id` | each film category points to one category name |

**Why not the other shared columns:**
- `last_update` appears in every table, but it is only a timestamp of when the row was
  last changed. Equal timestamps mean nothing, so it is not a key.
- `customer_id` and `staff_id` are shared by payments and rentals, but they are not
  unique per row: one customer has many payments and many rentals. Merging on them
  alone would match every payment with every rental of that customer and multiply
  the rows. `rental_id` links each payment to exactly one rental.

When merging, `last_update` columns would clash, so pandas would add suffixes
(`_x`, `_y`); selecting only the needed columns before merging avoids this.

## 3. Revenue by film attributes

### Q2. Revenue by film category
Total revenue, number of rentals and average amount per rental for each of the 16 categories.
Path: payment → rental → inventory → film_category → category.

In [14]:
revenue_by_category = (
    payments[["rental_id", "amount"]]
    .merge(rentals[["rental_id", "inventory_id"]], on="rental_id")
    .merge(inventory[["inventory_id", "film_id"]], on="inventory_id")
    .merge(film_category[["film_id", "category_id"]], on="film_id")
    .merge(category[["category_id", "name"]], on="category_id")
    .rename(columns={"name": "category"})
    .groupby("category")
    .agg(rentals=("rental_id", "count"), revenue=("amount", "sum"))
    .sort_values("revenue", ascending=False)
    .reset_index()
)

revenue_by_category["rentals_rank"]   = revenue_by_category["rentals"].rank(ascending=False).astype(int)
revenue_by_category["avg_per_rental"] = revenue_by_category["revenue"] / revenue_by_category["rentals"]
revenue_by_category["revenue_share"]  = revenue_by_category["revenue"] / total_revenue

# Sanity check: revenue covered by the merge vs total revenue
print(f"Revenue in merge: {revenue_by_category['revenue'].sum():,.2f}")
print(f"Total revenue:    {total_revenue:,.2f}")

revenue_by_category.style.format(
    {"revenue": "{:,.2f}", "avg_per_rental": "{:.2f}", "revenue_share": "{:.1%}"}
).hide(axis="index")

Revenue in merge: 67,406.56
Total revenue:    67,406.56


category,rentals,revenue,rentals_rank,avg_per_rental,revenue_share
Sports,1179,"5,314.21",1,4.51,7.9%
Sci-Fi,1101,"4,756.98",4,4.32,7.1%
Animation,1166,"4,656.30",2,3.99,6.9%
Drama,1060,"4,587.39",6,4.33,6.8%
Comedy,941,"4,383.58",11,4.66,6.5%
Action,1112,"4,375.85",3,3.94,6.5%
New,940,"4,351.62",12,4.63,6.5%
Games,969,"4,281.33",9,4.42,6.4%
Foreign,1033,"4,270.67",8,4.13,6.3%
Family,1096,"4,226.07",5,3.86,6.3%


**Observation:** Revenue is fairly evenly spread across the 16 categories (≈5–8% each).
Sports earns the most (5,314, ≈7.9%), about 55% more than Music (3,418, ≈5.1%).
Rental volume and revenue do not rank identically, because the average amount per rental
differs: Comedy (≈4.66) and New (≈4.63) rank low by rentals but high by revenue, while
Family (≈3.86), Children and Classics (≈3.87–3.88) earn the least per rental. Action is
3rd by rentals but only 6th by revenue.

**Data check:** the merged revenue equals total revenue (67,406.56), so every payment was
linked to exactly one category; no rows were lost or duplicated.

### Q3. Revenue by film rating
Does revenue by rating follow the number of films in each rating?
Path: film → inventory → rental → payment, plus the film count per rating for comparison.

In [15]:
films_per_rating = films["rating"].value_counts().rename_axis("rating").reset_index(name="films")

revenue_by_rating = (
    films[["film_id", "rating"]]
    .merge(inventory[["film_id", "inventory_id"]], on="film_id")
    .merge(rentals[["inventory_id", "rental_id"]], on="inventory_id")
    .merge(payments[["rental_id", "amount"]], on="rental_id")
    .groupby("rating")
    .agg(rentals=("rental_id", "count"), revenue=("amount", "sum"))
    .reset_index()
    .merge(films_per_rating, on="rating")
    .sort_values("revenue", ascending=False)
)

revenue_by_rating["avg_per_rental"] = revenue_by_rating["revenue"] / revenue_by_rating["rentals"]
revenue_by_rating["revenue_per_film"] = revenue_by_rating["revenue"] / revenue_by_rating["films"]
revenue_by_rating["films_share"]   = revenue_by_rating["films"] / revenue_by_rating["films"].sum()
revenue_by_rating["revenue_share"] = revenue_by_rating["revenue"] / total_revenue

print(f"Revenue in merge: {revenue_by_rating['revenue'].sum():,.2f}")
print(f"Total revenue:    {total_revenue:,.2f}")

revenue_by_rating.style.format(
    {"revenue": "{:,.2f}", "avg_per_rental": "{:.2f}", "revenue_per_film": "{:.1f}",
     "films_share": "{:.1%}", "revenue_share": "{:.1%}"}
).hide(axis="index")

Revenue in merge: 67,406.56
Total revenue:    67,406.56


rating,rentals,revenue,films,avg_per_rental,revenue_per_film,films_share,revenue_share
PG-13,3585,"15,259.16",223,4.26,68.4,22.3%,22.6%
NC-17,3293,"13,875.07",210,4.21,66.1,21.0%,20.6%
PG,3212,"13,337.91",194,4.15,68.8,19.4%,19.8%
R,3181,"13,270.19",195,4.17,68.1,19.5%,19.7%
G,2773,"11,664.23",178,4.21,65.5,17.8%,17.3%


**Observation:** Revenue by rating closely follows the size of the catalogue: each rating's
share of revenue (17.3–22.6%) is within about half a percentage point of its share of films.
PG-13 earns the most (15,259) because it has the most films (223), and G the least (11,664)
because it has the fewest (178). Revenue per film (≈66–69) and the average amount per rental
(4.15–4.26) are almost identical across ratings, so rating does not affect price or demand per film.

**Data check:** the merged revenue equals total revenue (67,406.56).

## 4. Stores

### Q4. Revenue per store
"Store" can be defined in two ways, and they answer different questions:
- **Customer's home store** (`customer.store_id`): which store's customer base is worth more?
- **Store that owns the rented copy** (`inventory.store_id`): which store's shelves earn more?

I calculated both, then cross-tabulated them to see whether customers rent from their home store.

In [16]:
# Version 1: store where the customer is registered
revenue_by_customer_store = (
    payments[["payment_id", "customer_id", "amount"]]
    .merge(customers[["customer_id", "store_id"]], on="customer_id")
    .rename(columns={"store_id": "customer_store"})
    .groupby("customer_store")
    .agg(payments=("payment_id", "count"), revenue=("amount", "sum"))
    .reset_index()
)
revenue_by_customer_store["revenue_share"] = revenue_by_customer_store["revenue"] / total_revenue

# Version 2: store that owns the rented DVD copy
revenue_by_copy_store = (
    inventory[["inventory_id", "store_id"]]
    .merge(rentals[["inventory_id", "rental_id"]], on="inventory_id")
    .merge(payments[["rental_id", "amount"]], on="rental_id")
    .rename(columns={"store_id": "copy_store"})
    .groupby("copy_store")
    .agg(rentals=("rental_id", "count"), revenue=("amount", "sum"))
    .reset_index()
)
revenue_by_copy_store["revenue_share"] = revenue_by_copy_store["revenue"] / total_revenue

money = {"revenue": "{:,.2f}", "revenue_share": "{:.1%}"}
display(revenue_by_customer_store.style.format(money).hide(axis="index"))
display(revenue_by_copy_store.style.format(money).hide(axis="index"))

customer_store,payments,revenue,revenue_share
1,8747,"36,997.53",54.9%
2,7297,"30,409.03",45.1%


copy_store,rentals,revenue,revenue_share
1,7923,"33,679.79",50.0%
2,8121,"33,726.77",50.0%


In [17]:
# Payments by customer's home store (rows) × store that owns the copy (columns)
store_check = (
    payments[["payment_id", "customer_id", "rental_id"]]
    .merge(customers[["customer_id", "store_id"]].rename(columns={"store_id": "customer_store"}), on="customer_id")
    .merge(rentals[["rental_id", "inventory_id"]], on="rental_id")
    .merge(inventory[["inventory_id", "store_id"]].rename(columns={"store_id": "copy_store"}), on="inventory_id")
)

display(pd.crosstab(store_check["customer_store"], store_check["copy_store"], margins=True, margins_name="Total"))

# Same table as row percentages: where each store's customers rent from
display(pd.crosstab(store_check["customer_store"], store_check["copy_store"], normalize="index").style.format("{:.1%}"))

customers["store_id"].value_counts().rename_axis("customer_store").reset_index(name="registered_customers")

copy_store,1,2,Total
customer_store,,,
1,4326,4421,8747
2,3597,3700,7297
Total,7923,8121,16044


copy_store,1,2
customer_store,,
1,49.5%,50.5%
2,49.3%,50.7%


,customer_store,registered_customers
0,1,326
1,2,273


**Observation:** The answer depends on how "store" is defined.
- By the customer's home store, store 1 earns more (36,998 vs 30,409, ≈55% of revenue),
  because more customers are registered there (326 vs 273).
- By the store that owns the rented copy, the two stores earn almost exactly the same
  (33,680 vs 33,727).
- The cross-tab shows why: customers rent from both stores about equally, whatever their
  home store (store 1 customers rent 51% of their DVDs from store 2; store 2 customers
  rent 49% from store 1).

For stock and shelf decisions, revenue by copy store is the right measure, and the two stores
perform the same. Both versions total 67,406.56.

## 5. Customers

### Q5. Top 10 customers by number of rentals
Are the customers who rent the most also the ones who spend the most?
Rentals and payments are aggregated per customer first; names are added at the end.

In [18]:
rentals_per_customer = rentals.groupby("customer_id").agg(rentals=("rental_id", "count"))
paid_per_customer    = payments.groupby("customer_id").agg(total_paid=("amount", "sum"))
customer_activity    = rentals_per_customer.merge(paid_per_customer, on="customer_id")

top_10_renters = (
    customer_activity
    .nlargest(10, "rentals", keep="all")   # keep="all" includes ties at 10th place
    .reset_index()
    .merge(customers[["customer_id", "first_name", "last_name"]], on="customer_id")
)
top_10_renters["avg_per_rental"] = top_10_renters["total_paid"] / top_10_renters["rentals"]

top_10_renters[["first_name", "last_name", "rentals", "total_paid", "avg_per_rental"]].style.format(
    {"total_paid": "{:.2f}", "avg_per_rental": "{:.2f}"}
).hide(axis="index")

first_name,last_name,rentals,total_paid,avg_per_rental
ELEANOR,HUNT,46,216.54,4.71
KARL,SEAL,45,221.55,4.92
CLARA,SHAW,42,195.58,4.66
MARCIA,DEAN,42,175.58,4.18
TAMMY,SANDERS,41,155.59,3.79
SUE,PETERS,40,154.60,3.87
WESLEY,BULL,40,177.60,4.44
RHONDA,KENNEDY,39,194.61,4.99
MARION,SNYDER,39,194.61,4.99
TIM,CARY,39,175.61,4.50


In [19]:
# Relationship between rentals and spending across all 599 customers
corr = customer_activity["rentals"].corr(customer_activity["total_paid"])
print(f"Correlation between number of rentals and total paid: {corr:.2f}")

Correlation between number of rentals and total paid: 0.87


**Observation:** The biggest renters are mostly also the biggest spenders: 8 of the top 10
customers by rentals are in the top 10 by total paid (notebook 1, Q13), and across all 599
customers the correlation between rentals and total paid is 0.87. Rental count is the main
driver of spending. The exceptions are TAMMY SANDERS and SUE PETERS, who rented 40–41 times
but paid only ~155 each (≈3.8–3.9 per rental vs ≈4.9 for KARL SEAL), so they mostly rent
cheaper titles.

## 6. Catalogue coverage

### Q6. Films that were never rented
A film can be unrented for two reasons: the store has no copy of it, or it has copies that
nobody rented. I checked both with left merges (the pandas version of `LEFT JOIN … IS NULL`).

In [20]:
# Reason 1: films with no copy in inventory
films_no_copies = (
    films[["film_id", "title"]]
    .merge(inventory[["film_id", "inventory_id"]], on="film_id", how="left", indicator=True)
    .query('_merge == "left_only"')
    [["film_id", "title"]]
    .sort_values("title")
)

# Reason 2: films with copies, but no rental of any copy
films_stocked_not_rented = (
    films[["film_id", "title"]]
    .merge(inventory[["film_id", "inventory_id"]], on="film_id")
    .merge(rentals[["inventory_id", "rental_id"]], on="inventory_id", how="left")
    .groupby(["film_id", "title"])
    .agg(rentals=("rental_id", "count"))   # count() skips NaN, so an unrented copy adds 0
    .query("rentals == 0")
)

print(f"Films with no copies in inventory:       {len(films_no_copies)}")
print(f"Films with copies but never rented:      {len(films_stocked_not_rented)}")
print(f"Total films never rented:                {len(films_no_copies) + len(films_stocked_not_rented)}")

films_no_copies.style.hide(axis="index")

Films with no copies in inventory:       42
Films with copies but never rented:      0
Total films never rented:                42


film_id,title
14,ALICE FANTASIA
33,APOLLO TEEN
36,ARGONAUTS TOWN
38,ARK RIDGEMONT
41,ARSENIC INDEPENDENCE
87,BOONDOCK BALLROOM
108,BUTCH PANTHER
128,CATCH AMISTAD
144,CHINATOWN GLADIATOR
148,CHOCOLATE DUCK


**Observation:** 42 of the 1,000 films were never rented, and all of them are films with no
copies in inventory (matching the SQL result). Every film the store actually stocks was rented
at least once. So unrented titles are a stock issue, not a demand issue: these 42 films cannot
be rented because the store does not own a copy.

## 7. Summary

**Key findings**
- **Categories:** Sports leads both revenue (5,314, ≈7.9%) and rentals (1,179), but revenue is
  fairly evenly spread across the 16 categories (≈5–8% each). Average price per rental differs:
  Comedy and New earn ≈4.6 per rental, Family, Children and Classics ≈3.9 (Q2).
- **Ratings:** revenue by rating follows catalogue size. PG-13 earns the most (15,259) because it
  has the most films (223); revenue per film and price per rental are almost the same for every
  rating (Q3).
- **Stores:** the answer depends on the definition. By home store, store 1 earns more (≈55%),
  because more customers are registered there; by the store that owns the copy, both earn the
  same (≈50% each). Customers do not stick to their home store: about half of their rentals
  come from the other store (Q4).
- **Customers:** renting more strongly goes with spending more (correlation 0.87 across all
  599 customers); 8 of the top 10 renters are also top-10 spenders. Exceptions such as
  TAMMY SANDERS and SUE PETERS rent often but choose cheaper titles (Q5).
- **Unrented films:** all 42 never-rented films have no copies in inventory, and every stocked
  film was rented at least once, so unrented titles are a stock issue, not a demand issue (Q6).

**Data checks**
- In every merge, the merged revenue matched total revenue (67,406.56), so no payments were
  lost or duplicated.
- Results match the earlier SQL queries (category revenue, 42 films without copies).